In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd
pd.set_option('display.max_rows', 8)
!date

Tue 06 Aug 2024 04:32:52 PM PDT


# Mean deaths and stillbirths averted by adding folate to rice in India in 2030 by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import gbd_mapping

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [3]:
location = "India"
asfr = vivarium_inputs.get_measure(gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location, years=2022).value

In [4]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel("parameter")

In [5]:
# Scale ASFR in each category down proportionally to the scale-down in TFR forecasted
asfr_2030_to_2022_ratio = (
    1.61 / 1.91 # http://ihmeuw.org/6j8s
)
asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2022        2023        0.000315
                  15.0       20.0     2022        2023        0.008079
                  20.0       25.0     2022        2023        0.087454
                  25.0       30.0     2022        2023        0.110737
                                                                ...   
                  35.0       40.0     2022        2023        0.028718
                  40.0       45.0     2022        2023        0.009527
                  45.0       50.0     2022        2023        0.002772
                  50.0       55.0     2022        2023        0.000257
Name: value, Length: 9, dtype: float64

In [6]:
asfr = asfr.reset_index().assign(year_start=2030, year_end=2031).set_index(asfr.index.names).value
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
India     Female  0.000000   0.019178   2030        2031        0.000000
                  0.019178   0.076712   2030        2031        0.000000
                  0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                                                                  ...   
                  35.000000  40.000000  2030        2031        0.028718
                  30.000000  35.000000  2030        2031        0.069523
                  20.000000  25.000000  2030        2031        0.087454
                  25.000000  30.000000  2030        2031        0.110737
Name: value, Length: 50, dtype: float64

In [7]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import get_location_id
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [8]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = get_location_id(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(data.drop("run_id", axis="columns").rename(columns={"population": "value"}), fill_value=None, cols_to_fill=utilities.DRAW_COLUMNS)
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(data, interval_column="age", split_column_prefix="age")
    data = utilities.split_interval(data, interval_column="year", split_column_prefix="year")
    return utilities.sort_hierarchical_data(data)


In [9]:
pop = get_population_future(location, 2030).value.reindex(asfr.index).fillna(0)
pop

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2030        2031        1.761904e+05
                  0.019178   0.076712    2030        2031        5.254291e+05
                  0.076712   0.500000    2030        2031        0.000000e+00
                  0.500000   1.000000    2030        2031        0.000000e+00
                                                                     ...     
          Male    80.000000  85.000000   2030        2031        6.771079e+06
                  85.000000  90.000000   2030        2031        2.880948e+06
                  90.000000  95.000000   2030        2031        9.755468e+05
                  95.000000  125.000000  2030        2031        2.694498e+05
Name: value, Length: 50, dtype: float64

In [10]:
# Messed with younger ages, but luckily none of these are WRA
pop[pop == 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  0.076712   0.5      2030        2031        0.0
                  0.500000   1.0      2030        2031        0.0
                  1.000000   2.0      2030        2031        0.0
                  2.000000   5.0      2030        2031        0.0
          Male    0.076712   0.5      2030        2031        0.0
                  0.500000   1.0      2030        2031        0.0
                  1.000000   2.0      2030        2031        0.0
                  2.000000   5.0      2030        2031        0.0
Name: value, dtype: float64

In [11]:
n_births = (pop * asfr).sum()
n_births

19609719.34389394

In [12]:
sbr = vivarium_inputs.get_measure(gbd_mapping.covariates.stillbirth_to_live_birth_ratio,
                                  "estimate", location, years=2022).value
sbr

location  year_start  year_end  parameter  
India     2022        2023      lower_value    0.016328
                                mean_value     0.016328
                                upper_value    0.016328
Name: value, dtype: float64

In [13]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel("parameter")
sbr

location  year_start  year_end
India     2022        2023        0.016328
Name: value, dtype: float64

In [14]:
sbr = sbr.values[0]

In [15]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

19.929907151540085

In [16]:
dist_births_and_stillbirths_by_wealth = pd.Series( # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
    dict(                          # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
        q1=56_979,
        q2=50_335,
        q3=45_189,
        q4=42_611,
        q5=36_290,
    )
)

s_births = n_births * dist_births_and_stillbirths_by_wealth / dist_births_and_stillbirths_by_wealth.sum()
s_births

q1    4.828535e+06
q2    4.265506e+06
q3    3.829422e+06
q4    3.610956e+06
q5    3.075300e+06
dtype: float64

In [17]:
s_births_and_stillbirths_by_wealth = births_and_stillbirths * dist_births_and_stillbirths_by_wealth / dist_births_and_stillbirths_by_wealth.sum()
s_births_and_stillbirths_by_wealth

q1    4.907375e+06
q2    4.335154e+06
q3    3.891949e+06
q4    3.669916e+06
q5    3.125514e+06
dtype: float64

In [18]:
ntd_deaths = 4_273.37 # http://ihmeuw.org/6j1h (for under-1 year olds)

# ntd_deaths *= 5 # sensitivity analysis, bring into line with Bhide et al systematic review


ntd_death_rate = ntd_deaths / n_births
10_000 * ntd_death_rate

2.1792101789211165

In [19]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69-51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (ntd_deaths + ntd_stillbirths) / n_births # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

8.353639019197614

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [20]:
s_baseline_folate = pd.Series(
    dict(
        q1=220,
        q2=220,
        q3=220,
        q4=220,
        q5=220, # NRV is 400 mcg/day
    )
)

In [21]:
s_dist_deaths_by_wealth = pd.Series( # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
    dict(                            # it includes wealth stratification, but has a very low threshold for insufficiency 
        q1=1,                        # so I am assuming that most everyone is in the danger zone for low folate
        q2=1,
        q3=1,
        q4=1,
        q5=1,
    )       
)

s_dist_deaths_by_wealth

q1    1
q2    1
q3    1
q4    1
q5    1
dtype: int64

In [22]:
s_ntd_death_rate = ntd_deaths/n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate

q1    2.17921
q2    2.17921
q3    2.17921
q4    2.17921
q5    2.17921
dtype: float64

In [23]:
s_ntd_death_count = s_ntd_death_rate * s_births_and_stillbirths_by_wealth
s_ntd_death_count

q1    1069.420132
q2     944.721078
q3     848.137495
q4     799.751860
q5     681.115088
dtype: float64

In [24]:
s_ntd_death_count.sum(), ntd_deaths # should be similar

(4343.145652958891, 4273.37)

In [25]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

q1    3030.023707
q2    2676.709722
q3    2403.056236
q4    2265.963603
q5    1929.826082
dtype: float64

In [26]:
s_ntd_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_count

q1    4099.443838
q2    3621.430801
q3    3251.193731
q4    3065.715463
q5    2610.941169
dtype: float64

In [27]:
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """
    
    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == 'daly':
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == 'crider':
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc
backcalc_rbc(s_ntd_count / s_births, 'daly')

q1    1283.442766
q2    1283.442766
q3    1283.442766
q4    1283.442766
q5    1283.442766
dtype: float64

In [28]:
backcalc_rbc(s_ntd_count / s_births, 'crider')  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9

q1    941.439754
q2    941.439754
q3    941.439754
q4    941.439754
q5    941.439754
dtype: float64

In [29]:
s_daily_rice = pd.Series( # Zeb and Alix analysis of HCES
    dict(
        q1=213.570675, # grams
        q2=175.027768,
        q3=163.699804,
        q4=163.363078,
        q5=127.874174,
    )
)

In [30]:
amount_of_folate_in_rice = 12.50/100 # mcg/g, from slide deck JG shared to ADF, credited to  Helena.Pachon@emory.edu

In [31]:
s_delta_folate_mcg = s_daily_rice * amount_of_folate_in_rice
s_delta_folate_mcg

q1    26.696334
q2    21.878471
q3    20.462476
q4    20.420385
q5    15.984272
dtype: float64

In [32]:
delta_folate_pct = 100 * s_delta_folate_mcg / s_baseline_folate
delta_folate_pct

q1    12.134697
q2     9.944760
q3     9.301125
q4     9.281993
q5     7.265578
dtype: float64

In [33]:
delta_rbc_pct = 6/10 * delta_folate_pct
delta_rbc_pct

q1    7.280818
q2    5.966856
q3    5.580675
q4    5.569196
q5    4.359347
dtype: float64

In [34]:
eff_fort_baseline_path = '../0100_data_prep/results/folate/rice/baseline_fortification/effective_coverage/india.csv'
eff_fort_intervention_path = '../0100_data_prep/results/folate/rice/intervention/intervention_fortification/effective_coverage/india.csv'

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
df_eff_fort_intervention = pd.read_csv(eff_fort_intervention_path)

In [35]:
quintile_name_map = {
    'lowest': 'q1',
    'second': 'q2',
    'middle': 'q3',
    'fourth': 'q4',
    'highest': 'q5',
}
df_eff_fort_baseline.index = df_eff_fort_baseline.wealth_quintile.map(quintile_name_map)
df_eff_fort_intervention.index = df_eff_fort_intervention.wealth_quintile.map(quintile_name_map)

In [36]:
delta_frac_fortified = df_eff_fort_intervention.value - df_eff_fort_baseline.value
delta_frac_fortified

wealth_quintile
q1    0.185658
q1    0.095485
q1    0.140290
q1    0.112683
        ...   
q5    0.243118
q5    0.244497
q5    0.260097
q5    0.263533
Name: value, Length: 500, dtype: float64

In [37]:
RBC_baseline = backcalc_rbc(s_ntd_count / s_births, 'crider')
RBC_baseline

q1    941.439754
q2    941.439754
q3    941.439754
q4    941.439754
q5    941.439754
dtype: float64

In [38]:
RBC_with_fort = RBC_baseline * (1 + delta_rbc_pct/100 * delta_frac_fortified)
RBC_with_fort

q1    954.165592
q1    947.984695
q1    951.055850
q1    949.163577
         ...    
q5    951.417480
q5    951.474083
q5    952.114290
q5    952.255300
Length: 500, dtype: float64

In [39]:
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == 'daly':
        ln_odds = 1.6563 - 1.2193*ln_rbc
    elif method == 'crider':
        ln_odds = 4.57 - 1.70*ln_rbc
    p = np.exp(ln_odds) # TODO: better transformation
    return p
s_ntd_rate_with_fort = calc_ntd_pr(RBC_with_fort, 'crider')
10_000 * s_ntd_rate_with_fort

q1    8.305493
q1    8.397761
q1    8.351713
q1    8.380038
        ...   
q5    8.346317
q5    8.345473
q5    8.335935
q5    8.333837
Length: 500, dtype: float64

In [40]:
s_pct_averted = 100 * (1 - s_ntd_rate_with_fort / (s_ntd_count / s_births))
s_pct_averted

q1    2.173660
q1    1.086874
q1    1.629259
q1    1.295632
        ...   
q5    1.692814
q5    1.702756
q5    1.815092
q5    1.839807
Length: 500, dtype: float64

In [41]:
s_ntd_count_with_fort = s_ntd_rate_with_fort * s_births
s_ntd_averted = s_ntd_count - s_ntd_count_with_fort
s_ntd_averted

q1    89.107991
q1    44.555796
q1    66.790547
q1    53.113716
        ...    
q5    44.198372
q5    44.457949
q5    47.390976
q5    48.036281
Length: 500, dtype: float64

In [42]:
10_000 * s_ntd_averted / s_births

q1    0.184545
q1    0.092276
q1    0.138325
q1    0.110000
        ...   
q5    0.143721
q5    0.144565
q5    0.154102
q5    0.156200
Length: 500, dtype: float64

In [43]:
# DALYs averted
tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()

In [44]:
yll_per_ntd = float(tmrle.iloc[0])
yll_per_ntd

89.95803974533831

In [45]:
s_ntd_averted * yll_per_ntd

q1    8015.980202
q1    4008.152050
q1    6008.346695
q1    4778.005739
         ...     
q5    3975.998912
q5    3999.349917
q5    4263.199315
q5    4321.249657
Length: 500, dtype: float64